In [ ]:
import pandas as pd 
import re

In [ ]:
def get_xml_data_descripteur(filename):
    doc = minidom.parse(filename) 
    root = doc.documentElement

    c_list = []
    
    c_nodes = get_xmlnode(root, 'DescriptorRecord')
    
    for node in c_nodes:
        #--------------Get descriptor ui
        node_d_ui = get_xmlnode(node, 'DescriptorUI')
        d_ui = get_nodevalue(node_d_ui[0])
        
        #--------------Get descriptor name
        node_des = get_xmlnode(node, 'DescriptorName')
        node_name = node_des.item(0).getElementsByTagName('String')
        c_name = get_nodevalue(node_name[0])
        
        #--------------Get tree number of this descriptor
        node_treeList = get_xmlnode(node, 'TreeNumberList')
        tree = []
        if len(node_treeList) == 0:
            tree.append('null') 
        else : 
            node_tree = node_treeList.item(0).getElementsByTagName('TreeNumber')
            for i in range(len(node_tree)):
                c_tree = get_nodevalue((node_tree[i]))
                tree.append(c_tree)
 
        #--------------Get UI of concept preferred of this descriptor
        node_concept = get_xmlnode(node, 'Concept')
        #print(node_concept[0].getAttribute('PreferredConceptYN')) #the first element of <Concept> is concept preferred
        node_ui = node_concept.item(0).getElementsByTagName('ConceptUI')
        c_ui = get_nodevalue(node_ui[0])
        
        #--------------Get relation of concept preferred of this descriptor
        node_rel = get_xmlnode(node, 'ConceptRelation')
        relList = []
        for i in range(len(node_rel)):
            rel_name = node_rel[i].getAttribute('RelationName')
            node_rel_ui2 = node_rel.item(i).getElementsByTagName('Concept2UI')
            c_rel_ui2 = get_nodevalue(node_rel_ui2[0])
            relList.append(rel_name)
            relList.append(c_rel_ui2)
        
        #--------------Get Scope Note of this descriptor
        node_scopeNote = get_xmlnode(node, 'ScopeNote')
        if len(node_scopeNote) == 0:
            c_scopeNote = ["null"]
        else:
            c_scopeNote = get_nodevalue(node_scopeNote[0])
            
        #--------------Get list of word of concept preferred.
        node_word_list = get_xmlnode(node, 'TermList')
        word_list = []
        for i in range(len(node_word_list)):
            word_ui = node_word_list.item(i).getElementsByTagName('TermUI')
            word = node_word_list.item(i).getElementsByTagName('String')
            word_note = node_word_list.item(i).getElementsByTagName('TermNote')
            wl_ui = get_nodevalue(word_ui[0])
            c_word = get_nodevalue(word[0])
            if len(word_note) == 0:
                w_note = "null"
            else:
                w_note = get_nodevalue(word_note[0])
            word_list.append(wl_ui)
            word_list.append(c_word) 
            word_list.append(w_note)
            
        concept = {}
        concept['c_ui'], concept['d_name'], concept['tree'], concept['scopeNote'], concept['d_ui'], concept['rel'], concept['word_list'] = (c_ui, re.findall(r'[[](.*?)[]]', c_name)[0].strip('['), tree, c_scopeNote, d_ui, relList, word_list)
    
        c_list.append(concept)
    return c_list   

In [ ]:

if __name__ == "__main__":
    
    filename  = '/Users/yaqi/Desktop/code_fusion_concept/Ontologie/MeSH/fredesc2019.xml'
    list_des = get_xml_data_descripteur(filename)
    df_descriptor = pd.DataFrame(list_des)
    #export_csv = df_descriptor.to_csv(r'data_MeSH.csv', index = None, header = True)
    #print(export_csv)